# Gun Detection Training (Local)
This notebook runs the full pipeline step by step with printed outputs per stage.

In [ ]:
import os
import json
import random
import shutil
from pathlib import Path

import cv2
import yaml
from tqdm import tqdm
from ultralytics import YOLO

try:
    import torch
    cuda_available = torch.cuda.is_available()
except Exception:
    cuda_available = False

DATASET_ROOT = Path("./Gun_Action_Recognition_Dataset")
OUTPUT_ROOT = Path("./outputs/yolo_dataset")
CHECKPOINT_DIR = Path("./outputs/checkpoints")
FINAL_MODELS_DIR = Path("./models")
RESULTS_DIR = Path("./outputs/test_results")

FRAME_SKIP = 5
FRAME_SKIP_NO_GUN = 50
TRAIN_SPLIT = 0.7

EPOCHS = 100
IMG_SIZE = 640
BATCH_SIZE = 8
MODEL_SIZE = "yolov8s"
DEVICE = 0

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Dataset root: {DATASET_ROOT.resolve()}")
print(f"Output root: {OUTPUT_ROOT.resolve()}")
print(f"Checkpoint dir: {CHECKPOINT_DIR.resolve()}")
print(f"Final models dir: {FINAL_MODELS_DIR.resolve()}")
print(f"Results dir: {RESULTS_DIR.resolve()}")
print(f"Frame skip: {FRAME_SKIP}")
print(f"Frame skip (No Gun): {FRAME_SKIP_NO_GUN}")
print(f"Train split: {TRAIN_SPLIT}")
print(f"Epochs: {EPOCHS}")
print(f"Image size: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Model size: {MODEL_SIZE}")
print(f"Device: {DEVICE}")
print(f"CUDA available: {cuda_available}")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def extract_and_convert(dataset_root, output_root, frame_skip=5, frame_skip_no_gun=50, train_split=0.8):
    """Extract frames and convert labels to YOLO format while limiting background frames."""
    output_root = Path(output_root)
    dataset_root = Path(dataset_root)

    for split in ["train", "val"]:
        (output_root / split / "images").mkdir(parents=True, exist_ok=True)
        (output_root / split / "labels").mkdir(parents=True, exist_ok=True)

    categories = ["Handgun", "Machine_Gun", "No_Gun"]
    class_mapping = {"Handgun": 0, "Machine_Gun": 1}

    all_videos = []
    for category in categories:
        category_path = dataset_root / category
        if category_path.exists():
            video_folders = [f for f in category_path.iterdir() if f.is_dir()]
            all_videos.extend([(category, vf) for vf in video_folders])

    random.shuffle(all_videos)
    split_idx = int(len(all_videos) * train_split)
    train_videos = all_videos[:split_idx]
    val_videos = all_videos[split_idx:]

    print("=" * 60)
    print("EXTRACTING FRAMES AND CONVERTING LABELS")
    print("=" * 60)
    print("\nDataset Split:")
    print(f"   Training: {len(train_videos)} videos")
    print(f"   Validation: {len(val_videos)} videos\n")

    total_images = 0
    total_annotations = 0
    total_backgrounds = 0

    for split_name, video_list in [("train", train_videos), ("val", val_videos)]:
        print(f"\n{'='*60}")
        print(f"Processing {split_name.upper()} videos...")
        print(f"{'='*60}\n")

        for category, video_folder in tqdm(video_list, desc=f"{split_name}"):
            video_path = video_folder / "video.mp4"
            if not video_path.exists():
                continue

            label_json = video_folder / "label.json"
            label_data = {}
            if label_json.exists():
                with open(label_json, "r") as f:
                    label_data = json.load(f)

            annotations_by_frame = {}
            if "annotations" in label_data:
                for ann in label_data["annotations"]:
                    img_id = ann.get("image_id", 0)
                    annotations_by_frame.setdefault(img_id, []).append(ann)

            cap = cv2.VideoCapture(str(video_path))
            frame_idx = 0

            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                current_skip = frame_skip_no_gun if category == "No_Gun" else frame_skip
                if frame_idx % current_skip == 0:
                    json_image_id = frame_idx + 1
                    has_boxes = False
                    yolo_lines = []

                    if json_image_id in annotations_by_frame:
                        height, width = frame.shape[:2]
                        for ann in annotations_by_frame[json_image_id]:
                            bbox = ann.get("bbox", [0, 0, 0, 0])
                            if len(bbox) == 4:
                                x, y, w, h = bbox
                                if w > 0 and h > 0:
                                    x_center = max(0, min(1, (x + w / 2) / width))
                                    y_center = max(0, min(1, (y + h / 2) / height))
                                    norm_w = max(0, min(1, w / width))
                                    norm_h = max(0, min(1, h / height))
                                    class_id = class_mapping.get(category, 0)
                                    yolo_lines.append(
                                        f"{class_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}\n"
                                    )
                                    has_boxes = True

                    should_save = False
                    if has_boxes:
                        should_save = True
                        total_annotations += len(yolo_lines)
                    elif category == "No_Gun":
                        should_save = True
                        total_backgrounds += 1
                    else:
                        if random.random() < 0.05:
                            should_save = True
                            total_backgrounds += 1

                    if should_save:
                        frame_name = f"{category}_{video_folder.name}_f{frame_idx:05d}"
                        img_path = output_root / split_name / "images" / f"{frame_name}.jpg"
                        label_path = output_root / split_name / "labels" / f"{frame_name}.txt"

                        cv2.imwrite(str(img_path), frame)
                        with open(label_path, "w") as f:
                            for line in yolo_lines:
                                f.write(line)
                        total_images += 1

                frame_idx += 1

            cap.release()

    print("\nEXTRACTION COMPLETE!")
    print(f"Total Saved Images: {total_images}")
    print(f"Total Bounding Boxes: {total_annotations}")
    if total_images > 0:
        pct = (total_backgrounds / total_images) * 100
        print(f"Total Background Images: {total_backgrounds} ({pct:.1f}% of dataset)")

In [ ]:
extract_and_convert(
    DATASET_ROOT,
    OUTPUT_ROOT,
    frame_skip=FRAME_SKIP,
    frame_skip_no_gun=FRAME_SKIP_NO_GUN,
    train_split=TRAIN_SPLIT,
)

In [ ]:
def create_yaml(output_root):
    output_root = Path(output_root)

    config = {
        "path": str(output_root.absolute()),
        "train": "train/images",
        "val": "val/images",
        "nc": 2,
        "names": {0: "Handgun", 1: "Machine_Gun"},
    }

    yaml_path = output_root / "gun_detection.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(config, f, default_flow_style=False)

    print("=" * 60)
    print("YAML CONFIGURATION CREATED")
    print("=" * 60)
    print(f"\nLocation: {yaml_path}\n")
    print("Contents:")
    print("-" * 40)
    with open(yaml_path, "r") as f:
        print(f.read())
    print("-" * 40)

    return yaml_path

In [ ]:
yaml_path = create_yaml(OUTPUT_ROOT)
print("\nConfiguration ready for training!")

In [ ]:
print("=" * 60)
print("STARTING YOLO TRAINING")
print("=" * 60)
print("\nTraining Configuration:")
print(f"   Model: {MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Image Size: {IMG_SIZE}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Config: {yaml_path}")
print(f"   Checkpoint Dir: {CHECKPOINT_DIR}\n")

try:
    import ultralytics.nn.tasks as ul_tasks
    import torch.serialization as torch_serialization
    torch_serialization.add_safe_globals([ul_tasks.DetectionModel])
except Exception as exc:
    print(f"Warning: could not register safe globals: {exc}")

last_checkpoint = CHECKPOINT_DIR / "gun_detection" / "weights" / "last.pt"

if last_checkpoint.exists():
    print("FOUND EXISTING CHECKPOINT!")
    print(f"   Resuming from: {last_checkpoint}\n")
    model = YOLO(str(last_checkpoint))
    results = model.train(resume=True)
else:
    print("Starting fresh training...\n")
    model = YOLO(f"{MODEL_SIZE}.pt")

    print("Starting training...\n")
    print("=" * 60)

    results = model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        patience=20,
        project=str(CHECKPOINT_DIR),
        name="gun_detection",
        save_period=5,
        device=DEVICE,
    )

print("\n" + "=" * 60)
print("TRAINING COMPLETE!")
print("=" * 60)
print("\nBest model saved to:")
print(f"   {CHECKPOINT_DIR}/gun_detection/weights/best.pt")
print("\nResults folder:")
print(f"   {CHECKPOINT_DIR}/gun_detection/")

In [ ]:
best_model = CHECKPOINT_DIR / "gun_detection" / "weights" / "best.pt"
if best_model.exists():
    shutil.copy(best_model, FINAL_MODELS_DIR / "gun_detection_best.pt")
    print("\nFINAL MODEL COPIED TO:")
    print(f"   {FINAL_MODELS_DIR / 'gun_detection_best.pt'}")
else:
    print("\nNo best model found yet.")

In [ ]:
print("=" * 60)
print("VALIDATING MODEL")
print("=" * 60)

best_model_path = FINAL_MODELS_DIR / "gun_detection_best.pt"
if not best_model_path.exists():
    print("Best model not found. Train or copy the model first.")
else:
    best_model = YOLO(str(best_model_path))
    metrics = best_model.val()

    print("\nValidation Metrics:")
    print(f"\n   mAP50: {metrics.box.map50:.4f}")
    print(f"   mAP50-95: {metrics.box.map:.4f}")
    print(f"   Precision: {metrics.box.mp:.4f}")
    print(f"   Recall: {metrics.box.mr:.4f}")

    print("\n   Per-Class AP50:")
    class_names = ["Handgun", "Machine_Gun"]
    for i, name in enumerate(class_names):
        print(f"      {name}: {metrics.box.ap50[i]:.4f}")

In [ ]:
TEST_VIDEO = ""

if not TEST_VIDEO:
    print("No test video set. Add a path to TEST_VIDEO to run inference.")
else:
    print(f"Testing on: {Path(TEST_VIDEO).name}\n")
    model = YOLO(str(FINAL_MODELS_DIR / "gun_detection_best.pt"))
    model.predict(
        source=str(TEST_VIDEO),
        save=True,
        conf=0.25,
        project=str(RESULTS_DIR),
        name="gun_detection",
        exist_ok=True,
    )
    print("\nTest complete!")
    print(f"Results saved in: {RESULTS_DIR / 'gun_detection'}")